# Ghost students

Students active in **≥3 of the first 4 weeks** of Autumn whose **later weekly average** activity is **< 10%** of their **early weekly average**.
Activity = LMS rows + video rows. Weeks with 0 activity count in the averages.

In [27]:
import pandas as pd

# Run from the Monster folder. Switch to Test/Predict by changing these two.
DATA_DIR = "Dataset/Training"
SPLIT = "training"  # filename prefix: la_mmh_<SPLIT>_..._fact.csv

ACADEMIC_YEAR = 2023      # 2 Oct 2023 – 28 Sep 2024, weeks 1–52
EARLY_WEEKS = range(1, 5)  # weeks 1–4 of Autumn
LATER_WEEKS = range(5, 53) # rest of the academic year
MIN_EARLY_ACTIVE = 3
DROP_RATIO = 0.10

## 1. Load activity rows

In [28]:
# Only the columns we need. The LMS file is ~640 MB, so this keeps memory down.
lms = pd.read_csv(f"{DATA_DIR}/la_mmh_{SPLIT}_lms_item_activity_by_session_fact.csv",
                  usecols=["PIT_ID", "P_ID", "DATE"])
# Duplicate LMS rows are exact copies sharing the same PIT_ID
lms = lms.drop_duplicates("PIT_ID")

video = pd.read_csv(f"{DATA_DIR}/la_mmh_{SPLIT}_video_activity_by_view_fact.csv")
# Drop exact duplicate rows only; a few PVT_IDs repeat with different values, so those are kept
video = video.drop_duplicates()

# One table, one row per LMS session or video view. SOURCE keeps them distinguishable.
activity = pd.concat([
    lms[["P_ID", "DATE"]].assign(SOURCE="LMS"),
    video[["P_ID", "DATE"]].assign(SOURCE="VIDEO"),
], ignore_index=True)

print(f"LMS rows:   {len(lms):,}")
print(f"Video rows: {len(video):,}")
print(f"Total:      {len(activity):,}")

LMS rows:   7,794,107
Video rows: 1,138,389
Total:      8,932,496


## 2. Dates → week numbers

In [29]:
calendar = pd.read_csv(f"{DATA_DIR}/la_mmh_calendar_dim.csv")
calendar = calendar.loc[calendar["ACADEMIC_YEAR"] == ACADEMIC_YEAR, ["DATE", "WEEK_NUMBER"]]

# Inner join: rows dated outside the academic year have no week and drop out
activity_wk = activity.merge(calendar, on="DATE", how="inner")

print(f"Rows outside academic year dropped: {len(activity) - len(activity_wk):,}")
print(f"Weeks: {activity_wk['WEEK_NUMBER'].min()}–{activity_wk['WEEK_NUMBER'].max()}")

Rows outside academic year dropped: 8,237
Weeks: 1–52


## 3. Rows per student per week

In [30]:
# One row per student, one column per week. reindex adds any week nobody was active in;
# fill_value=0 makes empty weeks 0 so they count in the averages.
weekly = (activity_wk.groupby(["P_ID", "WEEK_NUMBER"]).size()
          .unstack(fill_value=0)
          .reindex(columns=range(1, 53), fill_value=0))

print(f"Students: {len(weekly):,}")
weekly.head()

Students: 13,211


WEEK_NUMBER,1,2,3,4,5,6,7,8,9,10,...,43,44,45,46,47,48,49,50,51,52
P_ID,,,,,,,,,,,,,,,,,,,,,
P00000002,14,33,9,115,62,32,64,60,37,109,...,0,0,0,0,0,0,0,0,0,0
P00000003,0,0,0,0,0,1,4,0,7,34,...,0,0,0,4,1,0,4,4,0,0
P00000005,0,53,79,98,80,22,108,50,31,10,...,22,72,59,22,0,0,0,0,0,0
P00000006,86,71,47,22,33,24,15,15,54,12,...,6,0,0,0,0,0,0,0,0,0
P00000007,3,34,33,31,36,11,72,33,43,60,...,0,0,2,0,0,0,0,0,0,0


## 4. Early vs later activity

In [31]:
early = weekly[list(EARLY_WEEKS)]
later = weekly[list(LATER_WEEKS)]

# mean() divides by all weeks in the range, including the 0 weeks
stats = pd.DataFrame({
    "early_active_weeks": (early > 0).sum(axis=1),
    "early_avg": early.mean(axis=1),
    "later_avg": later.mean(axis=1),
})
stats["ratio"] = stats["later_avg"] / stats["early_avg"]

qualified = stats["early_active_weeks"] >= MIN_EARLY_ACTIVE
dropped_off = stats["later_avg"] < DROP_RATIO * stats["early_avg"]
ghosts = stats[qualified & dropped_off].sort_values("ratio")

print(f"Active ≥{MIN_EARLY_ACTIVE} of first 4 weeks: {qualified.sum():,}")
print(f"Ghost students:                {len(ghosts):,} ({len(ghosts) / qualified.sum():.1%} of those)")

Active ≥3 of first 4 weeks: 8,586
Ghost students:                149 (1.7% of those)


## 5. Results

In [32]:
# All ghost students, lowest ratio first = biggest drop-off.
# option_context lifts pandas' row limit so none are hidden behind "..."
with pd.option_context("display.max_rows", None):
    display(ghosts.round(2))

,early_active_weeks,early_avg,later_avg,ratio
P_ID,,,,
P00001439,3,16.25,0.00,0.00
P00007556,3,12.50,0.00,0.00
P00001380,3,4.00,0.00,0.00
P00012742,3,1.50,0.00,0.00
P00010403,3,3.50,0.02,0.01
P00013302,3,6.25,0.04,0.01
P00007530,4,5.25,0.04,0.01
P00012501,3,25.50,0.58,0.02
P00008016,3,32.50,0.79,0.02


In [33]:
# Just the IDs, comma-separated for copying
ghost_ids = ghosts.index.tolist()
print(f"{len(ghost_ids)} ghost P_IDs:\n")
print(", ".join(ghost_ids))

149 ghost P_IDs:

P00001439, P00007556, P00001380, P00012742, P00010403, P00013302, P00007530, P00012501, P00008016, P00016487, P00002518, P00012874, P00013658, P00011392, P00005932, P00007755, P00004760, P00009295, P00001970, P00010917, P00016227, P00004666, P00016793, P00003375, P00008161, P00005247, P00008340, P00016590, P00013120, P00010209, P00649204, P00009826, P00002812, P00010032, P00012618, P00009941, P00001068, P00006611, P00012347, P00013185, P00004187, P00017145, P00018125, P00001621, P00003612, P00014093, P00009904, P00005862, P00001394, P00005617, P00001423, P00016383, P00005766, P00014282, P00014866, P00002223, P00001155, P00003067, P00005021, P00017961, P00006770, P00011991, P00003646, P00008242, P00013829, P00009217, P00018152, P00015303, P00013892, P00011815, P00001454, P00008231, P00017535, P00013924, P00006467, P00005733, P00005844, P00006485, P00001271, P00004980, P00012152, P00015613, P00013793, P00015144, P00010769, P00009685, P00007912, P00001113, P00017094, P00

In [34]:
# One ID per line: copy the output and paste into Excel to fill a single column
print("\n".join(ghost_ids))

P00001439
P00007556
P00001380
P00012742
P00010403
P00013302
P00007530
P00012501
P00008016
P00016487
P00002518
P00012874
P00013658
P00011392
P00005932
P00007755
P00004760
P00009295
P00001970
P00010917
P00016227
P00004666
P00016793
P00003375
P00008161
P00005247
P00008340
P00016590
P00013120
P00010209
P00649204
P00009826
P00002812
P00010032
P00012618
P00009941
P00001068
P00006611
P00012347
P00013185
P00004187
P00017145
P00018125
P00001621
P00003612
P00014093
P00009904
P00005862
P00001394
P00005617
P00001423
P00016383
P00005766
P00014282
P00014866
P00002223
P00001155
P00003067
P00005021
P00017961
P00006770
P00011991
P00003646
P00008242
P00013829
P00009217
P00018152
P00015303
P00013892
P00011815
P00001454
P00008231
P00017535
P00013924
P00006467
P00005733
P00005844
P00006485
P00001271
P00004980
P00012152
P00015613
P00013793
P00015144
P00010769
P00009685
P00007912
P00001113
P00017094
P00005594
P00005251
P00016660
P00014528
P00008259
P00016013
P00006797
P00012341
P00007520
P00007793
P00013634
